# Movie Recommendation System - Exploratory Data Analysis (EDA)

This notebook explores the MovieLens 1M dataset to understand the distribution of ratings,
user demographics, and movie genres. The insights gained here will directly inform our
feature engineering phase.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Define data directory
DATA_DIR = "../data/ml-1m"

## 1. Load the Data
The 1M dataset uses `::` as a separator and has no header row.

In [ ]:
# Load Ratings
ratings_cols = ['userId', 'movieId', 'rating', 'timestamp']
ratings = pd.read_csv(os.path.join(DATA_DIR, 'ratings.dat'), sep='::', engine='python', names=ratings_cols, encoding='latin-1')

# Load Movies
movies_cols = ['movieId', 'title', 'genres']
movies = pd.read_csv(os.path.join(DATA_DIR, 'movies.dat'), sep='::', engine='python', names=movies_cols, encoding='latin-1')

# Load Users
users_cols = ['userId', 'gender', 'age', 'occupation', 'zipCode']
users = pd.read_csv(os.path.join(DATA_DIR, 'users.dat'), sep='::', engine='python', names=users_cols, encoding='latin-1')

print(f"Ratings: {ratings.shape[0]:,} rows")
print(f"Movies: {movies.shape[0]:,} rows")
print(f"Users: {users.shape[0]:,} rows")

## 2. Ratings Distribution
Let's see how users tend to rate movies. Are they generous or critical?

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=ratings, x='rating', palette='viridis')
plt.title('Distribution of Movie Ratings')
plt.xlabel('Rating (Stars)')
plt.ylabel('Count')
plt.show()

print("Mean Rating:", ratings['rating'].mean())
print("Median Rating:", ratings['rating'].median())

## 3. User Demographics
Analyzing the Age and Gender distribution of our users.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender
sns.countplot(data=users, x='gender', palette='Set2', ax=axes[0])
axes[0].set_title('User Gender Distribution (M=Male, F=Female)')

# Age
# Note: Age is categorized in the dataset (e.g., 1="Under 18", 25="25-34")
sns.countplot(data=users, x='age', palette='magma', ax=axes[1])
axes[1].set_title('User Age Distribution (Categorized)')

plt.tight_layout()
plt.show()

## 4. Sparsity Check
In recommendation systems, sparsity is a massive challenge. Let's calculate how sparse our user-item matrix is.

In [ ]:
total_possible_interactions = users.shape[0] * movies.shape[0]
actual_interactions = ratings.shape[0]
sparsity = 1 - (actual_interactions / total_possible_interactions)

print(f"Total possible interactions: {total_possible_interactions:,}")
print(f"Actual interactions (Ratings): {actual_interactions:,}")
print(f"Matrix Sparsity: {sparsity * 100:.2f}%")

**Observation:** A sparsity of ~95% means the vast majority of users have not rated the vast majority of movies. 
This confirms we need models capable of handling sparse matrices (like Matrix Factorization/Embeddings).

## 5. Most Popular Genres
Movies can have multiple genres separated by a pipe `|`. We need to explode them to count.

In [ ]:
# Split the '|' string and explode it into separate rows in one clean step
movies_exploded = movies.assign(genres=movies['genres'].str.split('|')).explode('genres')

plt.figure(figsize=(12, 6))
sns.countplot(data=movies_exploded, y='genres', order=movies_exploded['genres'].value_counts().index, palette='crest')
plt.title('Most Common Movie Genres')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.show()